# Baseline Experiment

## 1. Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import models, transforms
from medmnist import PathMNIST
import numpy as np
import random
import copy

import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

cuda


d:\Bachelorarbeit\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 2. Reproduzierbarkeit

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Nutze Gerät: {DEVICE}")

NUM_SAMPLES_PER_CLASS = 50
NUM_EPOCHS_FINAL = 50 # Wie lange das finale Modell trainiert wird

# 3. Datenvorbereitung & Transformationen

In [ ]:
data_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Wir nutzen nun `data_transform` für alle drei Splits
train_dataset = PathMNIST(split='train', transform=data_transform, download=True, size=224)
val_dataset = PathMNIST(split='val', transform=data_transform, download=True, size=224)
test_dataset = PathMNIST(split='test', transform=data_transform, download=True, size=224)

def get_balanced_subset(dataset, n_per_class):
    """ Erstellt einen kleinen, balancierten Sub-Datensatz """
    labels = dataset.labels.flatten()
    indices = []
    for i in np.unique(labels):
        class_indices = np.where(labels == i)[0]
        selected_indices = np.random.choice(class_indices, n_per_class, replace=False)
        indices.extend(selected_indices)
    return Subset(dataset, indices)

# Erstelle kleine Trainingsmenge
small_train_ds = get_balanced_subset(train_dataset, NUM_SAMPLES_PER_CLASS)

# 4. Cross Validierung

In [ ]:
print("\n--- STARTE PHASE A: OPTUNA TUNING ---")

def objective(trial):
    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32])
    
    target_labels = [train_dataset.labels[i][0] for i in small_train_ds.indices]
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    
    fold_accuracies = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(target_labels)), target_labels)):
        fold_train_ds = Subset(small_train_ds, train_idx)
        fold_val_ds = Subset(small_train_ds, val_idx)
        
        loader_train = DataLoader(fold_train_ds, batch_size=batch_size, shuffle=True)
        loader_val = DataLoader(fold_val_ds, batch_size=batch_size, shuffle=False)

        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        model.fc = nn.Linear(model.fc.in_features, 9)
        model = model.to(DEVICE)
        
        optimizer = optim.Adam(model.parameters(), lr=lr)
        criterion = nn.CrossEntropyLoss()

        best_acc = 0
        for epoch in range(15): # Kurzes Training für die Suche
            model.train()
            for inputs, targets in loader_train:
                inputs, targets = inputs.to(DEVICE), targets.flatten().to(DEVICE).long()
                optimizer.zero_grad()              
                outputs = model(inputs)            
                loss = criterion(outputs, targets) 
                loss.backward()                    
                optimizer.step()                   
            
            # Validierung (Muss INNERHALB der Epochen-Schleife sein!)
            model.eval()
            all_preds, all_targets = [], []
            with torch.no_grad():
                for inputs, targets in loader_val:
                    inputs, targets = inputs.to(DEVICE), targets.flatten().to(DEVICE).long()
                    _, predicted = torch.max(model(inputs), 1)
                    all_preds.extend(predicted.cpu().numpy())
                    all_targets.extend(targets.cpu().numpy())
                
            acc = accuracy_score(all_targets, all_preds)
            if acc > best_acc:
                best_acc = acc
            
        fold_accuracies.append(best_acc)

    return np.mean(fold_accuracies)

study = optuna.create_study(direction="maximize")
# Für einen echten Durchlauf auf n_trials=20 oder 30 setzen.
study.optimize(objective, n_trials=10) 

best_lr = study.best_params["lr"]
best_bs = study.best_params["batch_size"]

print(f"\nOptuna beendet. Beste Parameter gefunden:")
print(f"  Learning Rate: {best_lr:.6f}")
print(f"  Batch Size:    {best_bs}")

# 5. Finales Training

In [ ]:

print("\n--- STARTE PHASE B: FINALES TRAINING ---")
print(f"Trainiere auf vollen {len(small_train_ds)} Samples mit den besten Parametern...")

# Wir nutzen jetzt die KOMPLETTEN 450 Bilder fürs Training und das ECHTE Val-Set fürs Checkpointing
final_train_loader = DataLoader(small_train_ds, batch_size=best_bs, shuffle=True)
final_val_loader = DataLoader(val_dataset, batch_size=best_bs, shuffle=False)

final_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
final_model.fc = nn.Linear(final_model.fc.in_features, 9)
final_model = final_model.to(DEVICE)

optimizer_final = optim.Adam(final_model.parameters(), lr=best_lr)
criterion_final = nn.CrossEntropyLoss()

best_final_val_acc = 0.0
best_final_weights = None

for epoch in range(NUM_EPOCHS_FINAL):
    final_model.train()
    running_loss = 0.0
    for inputs, targets in final_train_loader:
        inputs, targets = inputs.to(DEVICE), targets.flatten().to(DEVICE).long()
        optimizer_final.zero_grad()
        outputs = final_model(inputs)
        loss = criterion_final(outputs, targets)
        loss.backward()
        optimizer_final.step()
        running_loss += loss.item()
        
    # Validierung auf dem echten MedMNIST Validation-Set
    final_model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for inputs, targets in final_val_loader:
            inputs, targets = inputs.to(DEVICE), targets.flatten().to(DEVICE).long()
            _, predicted = torch.max(final_model(inputs), 1)
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())
            
    acc = accuracy_score(all_targets, all_preds)
    
    if acc > best_final_val_acc:
        best_final_val_acc = acc
        best_final_weights = copy.deepcopy(final_model.state_dict())
        print(f"Epoch {epoch+1}/{NUM_EPOCHS_FINAL}: Neues bestes Modell! (Val Acc: {acc*100:.2f}%)")

# Lade das beste Modell für den finalen Test
final_model.load_state_dict(best_final_weights)
print("Finales Training abgeschlossen. Bestes Modell geladen.")

# 6. Finale Evaluierung auf dem Test-Set

In [ ]:
print("\n--- STARTE PHASE C: FINALE AUSWERTUNG AUF TESTDATEN ---")
test_loader = DataLoader(test_dataset, batch_size=best_bs, shuffle=False)

final_model.eval()
all_preds, all_targets, all_probs = [], [], []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(DEVICE), targets.flatten().to(DEVICE).long()
        outputs = final_model(inputs)
        
        probs = F.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

# Metriken berechnen
test_acc = accuracy_score(all_targets, all_preds)
test_bal_acc = balanced_accuracy_score(all_targets, all_preds)
test_prec = precision_score(all_targets, all_preds, average='macro', zero_division=0)
test_rec = recall_score(all_targets, all_preds, average='macro', zero_division=0)
test_f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
test_auc = roc_auc_score(all_targets, all_probs, multi_class='ovr', average='macro')

print("="*50)
print("BASELINE ERGEBNISSE (TEST-SET)")
print("="*50)
print(f"Accuracy:      {test_acc*100:.2f}%")
print(f"Balanced Acc:  {test_bal_acc*100:.2f}%")
print(f"Precision:     {test_prec*100:.2f}%")
print(f"Recall:        {test_rec*100:.2f}%")
print(f"F1-Score:      {test_f1*100:.2f}%")
print(f"AUC-ROC:       {test_auc*100:.2f}%")
print("="*50)